# Walt falando — LongCat-Video-Avatar 1.5 no Colab (Google AI Pro)
Gera os clipes do Walt (avatar IA) com boca sincronizada. Só GPU do Colab, nada de Modal.

1. *Ambiente de execução > Alterar o tipo* → **G4** (RTX PRO 6000, 96 GB) ou **A100**. A L4 não aguenta o modelo. 
2. *Ambiente de execução > Executar tudo*.
3. Os clipes vão para o Google Drive em `MyDrive/walt-clipes/` e ficam também para download no fim.

In [ ]:
#@title Configuração
REPO = "https://github.com/viniciushoscher-ctrl/estudio-avatar"
BRANCH = "claude/project-thread-fa7cvo"
IMAGEM = "avatares/walt-rosto.png"          # Walt à bancada
PROMPT_ARQ = "avatares/walt-prompt.txt"
FALAS_DIR = "walt/falas"                     # um .wav por clipe
SEED = 42
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import subprocess, torch
mem = torch.cuda.get_device_properties(0).total_memory/1e9
RESOLUCAO = "720p" if mem > 70 else "480p"   # 720p precisa de 80 GB (A100-80GB/H100)
print(f"GPU {torch.cuda.get_device_name(0)} {mem:.0f} GB -> {RESOLUCAO}")

In [ ]:
%%bash
set -e
cd /content
[ -d estudio-avatar ] || git clone -q --depth 1 -b claude/project-thread-fa7cvo https://github.com/viniciushoscher-ctrl/estudio-avatar
[ -d longcat ] || (GIT_LFS_SKIP_SMUDGE=1 git clone -q https://huggingface.co/spaces/victor/LongCat-Video-Avatar-1.5 longcat && cd longcat && git checkout -q daeb7c18bdfd051b5f94a2e88287a50ccf2988e3 && rm -rf assets)
apt-get -qq install -y ffmpeg > /dev/null
pip -q install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128
pip -q install xformers==0.0.31.post1 --index-url https://download.pytorch.org/whl/cu128
pip -q install "gradio==6.10.0" "spaces>=0.30" "huggingface_hub>=0.27,<1.0" hf_transfer transformers==4.49.0 diffusers==0.32.2 "accelerate>=1.0" "safetensors>=0.4" "einops>=0.8" loguru ftfy regex imageio imageio-ffmpeg opencv-python-headless Pillow numpy==1.26.4 scipy==1.15.3 scikit-image==0.25.2 soundfile==0.13.1 librosa==0.11.0 pyloudnorm==0.1.1 audio-separator==0.30.2 onnx==1.18.0 onnxruntime==1.20.1
echo OK

**Se pedir para reiniciar a sessão depois da instalação, reinicie e rode de novo a partir da célula abaixo.**

In [ ]:
# pesos completos bf16 (o INT8 deixava dentes verdes)
p = '/content/longcat/app.py'; s = open(p).read()
if '_M.from_pretrained' not in s:
    s = s.replace('"base_model_int8/*",', '"base_model_int8/*", "base_model/*",')
    s = s.replace('dit = load_quantized_dit(str(AVATAR_DIR), subfolder="base_model_int8", cp_split_hw=CP_SPLIT_HW)',
      'from longcat_video.modules.avatar.longcat_video_dit_avatar import LongCatVideoAvatarTransformer3DModel as _M\n'
      'dit = _M.from_pretrained(str(AVATAR_DIR), subfolder="base_model", cp_split_hw=CP_SPLIT_HW, torch_dtype=torch_dtype, '
      'enable_flashattn2=False, enable_flashattn3=False, enable_bsa=False, enable_xformers=True)')
    s = s.replace('if not (AVATAR_DIR / "base_model_int8" / "config.json").exists():',
                  'if not (AVATAR_DIR / "base_model" / "config.json").exists():')
    assert '_M.from_pretrained' in s
    open(p, 'w').write(s)
print('patch ok')

In [ ]:
import os, sys, time
os.environ.update(WEIGHTS_DIR='/content/pesos', HF_HUB_ENABLE_HF_TRANSFER='1', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
os.chdir('/content/longcat'); sys.path.insert(0, '/content/longcat')
import torch
import longcat_video.pipeline_longcat_video_avatar as P
class TextoNaCPU:
    def __init__(self, m): self.m, self.config, self.dtype = m.eval(), m.config, m.dtype
    def to(self, *a, **k): return self
    def __call__(self, ids, mask):
        with torch.no_grad(): return self.m(ids.cpu(), mask.cpu())
_mover = P.LongCatVideoAvatarPipeline.to
def _mover_sem_t5(pipe, device):
    t5, pipe.text_encoder = pipe.text_encoder, None
    _mover(pipe, device)
    pipe.text_encoder = t5 if isinstance(t5, TextoNaCPU) else TextoNaCPU(t5)
    return pipe
P.LongCatVideoAvatarPipeline.to = _mover_sem_t5
t = time.time(); import app as lc
print(f'modelo pronto em {time.time()-t:.0f}s')

In [ ]:
import math, tempfile, numpy as np
from pathlib import Path
from PIL import Image
FPS, QUADROS, COND = 25, 125, 13
def gerar(img_path, aud_path, prompt, destino, seed=SEED, resolucao=RESOLUCAO):
    tmp = Path(tempfile.mkdtemp())
    fala, sr = lc._load_audio_16k(str(aud_path)); dur = len(fala)/sr
    qf = math.ceil(dur*FPS); trechos = 1 + max(0, math.ceil((qf-QUADROS)/(QUADROS-COND)))
    total = QUADROS + (trechos-1)*(QUADROS-COND)
    emb = lc._prepare_audio_embedding(str(aud_path), lc.VOCAL_MODE_FAST, total, FPS, 1, lambda *a, **k: None)
    lc._configure_dit_acceleration("Exact 8-step")
    gen = torch.Generator(device=lc.device).manual_seed(int(seed))
    foto = Image.open(img_path).convert('RGB')
    with torch.inference_mode():
        saida, lat = lc.pipe.generate_ai2v(image=foto, prompt=prompt, negative_prompt=lc.NEGATIVE_PROMPT, resolution=resolucao,
            num_frames=QUADROS, num_inference_steps=8, text_guidance_scale=1.0, audio_guidance_scale=1.0,
            output_type='both', generator=gen, audio_emb=emb[:, :QUADROS], use_distill=True)
        atual = [Image.fromarray((q*255).astype(np.uint8)) for q in saida[0]]; todos = list(atual)
        ref = lat[:, :, :1].clone(); w, h = atual[0].size; ini = 0
        del saida; torch.cuda.empty_cache()
        for i in range(1, trechos):
            ini += QUADROS-COND
            saida, lat = lc.pipe.generate_avc(video=atual, video_latent=lat, prompt=prompt, negative_prompt=lc.NEGATIVE_PROMPT,
                height=h, width=w, num_frames=QUADROS, num_cond_frames=COND, num_inference_steps=8, text_guidance_scale=1.0,
                audio_guidance_scale=1.0, generator=gen, output_type='both', use_kv_cache=True, offload_kv_cache=True,
                enhance_hf=False, audio_emb=emb[:, ini:ini+QUADROS], ref_latent=ref, ref_img_index=10, mask_frame_range=3, use_distill=True)
            atual = [Image.fromarray((q*255).astype(np.uint8)) for q in saida[0]]; todos.extend(atual[COND:])
            del saida; torch.cuda.empty_cache()
    q = np.stack([np.asarray(x) for x in todos[:qf]])
    final = lc._save_video_ffmpeg_fast(q, tmp/'clip', str(aud_path), fps=FPS, quality=8)
    Path(destino).write_bytes(Path(final).read_bytes())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
base = Path('/content/estudio-avatar'); out = Path('/content/drive/MyDrive/walt-clipes'); out.mkdir(parents=True, exist_ok=True)
prompt = (base/PROMPT_ARQ).read_text().strip()
falas = sorted((base/FALAS_DIR).glob('*.wav'))
print(len(falas), 'falas')
for f in falas:
    dest = out/f.with_suffix('.mp4').name
    if dest.exists(): print('já existe', dest.name); continue
    t = time.time(); gerar(base/IMAGEM, f, prompt, dest)
    print(f'{dest.name} pronto em {time.time()-t:.0f}s')

In [ ]:
!cd /content/drive/MyDrive && zip -qr /content/walt-clipes.zip walt-clipes && ls -la walt-clipes
from google.colab import files; files.download('/content/walt-clipes.zip')